In [0]:
from pyspark.sql import functions as F
import uuid

CATALOG     = "claims"
AUDIT_TABLE = f"{CATALOG}.gold.dq_audit"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
    run_id STRING, layer STRING, table_name STRING, check_name STRING,
    severity STRING, passed BOOLEAN, observed STRING, expected STRING,
    checked_at TIMESTAMP)
""")

RUN_ID, _results = str(uuid.uuid4()), []

def check(layer, table, name, passed, observed, expected, severity="critical"):
    passed = bool(passed)
    _results.append((RUN_ID, layer, table, name, severity,
                     passed, str(observed), str(expected)))
    print(f"[{'PASS' if passed else 'FAIL'}] {table}.{name}  "
          f"observed={observed}  expected={expected}")

def flush():
    if not _results:
        return
    schema = ("run_id string, layer string, table_name string, check_name string, "
              "severity string, passed boolean, observed string, expected string")
    (spark.createDataFrame(_results, schema)
        .withColumn("checked_at", F.current_timestamp())
        .write.mode("append").saveAsTable(AUDIT_TABLE))
    print(f"wrote {len(_results)} results")

In [0]:
hdr   = spark.table("claims.silver.claim_header")
lines = spark.table("claims.silver.service_line")
bene  = spark.table("claims.silver.beneficiary")

t, u = hdr.count(), hdr.select("claim_id").distinct().count()
check("silver", "claim_header", "claim_id_unique", t == u, u, t)

orphans = hdr.join(bene.select("beneficiary_id"), "beneficiary_id", "left_anti").count()
check("silver", "claim_header", "fk_beneficiary_exists", orphans == 0, orphans, 0)

bad_span = hdr.filter(F.col("service_end") < F.col("service_start")).count()
check("silver", "claim_header", "service_dates_ordered", bad_span == 0, bad_span, 0)

post_mortem = (hdr.join(bene, "beneficiary_id")
    .filter(F.col("death_date").isNotNull() &
            (F.col("service_start") > F.col("death_date"))).count())
check("silver", "claim_header", "no_service_after_death",
      post_mortem == 0, post_mortem, 0, severity="warn")

neg = lines.filter(F.col("line_paid") < 0).count()
check("silver", "service_line", "payment_non_negative", neg == 0, neg, 0)

over = lines.filter(F.col("line_paid") > F.col("line_allowed")).count()
check("silver", "service_line", "paid_within_allowed", over == 0, over, 0, severity="warn")

bad_hcpcs = lines.filter(~F.col("hcpcs_code").rlike("^[A-Z0-9]{5}$")).count()
check("silver", "service_line", "hcpcs_format_valid",
      bad_hcpcs == 0, bad_hcpcs, 0, severity="warn")

zero_rate = lines.select(F.avg(F.col("is_zero_pay").cast("int")) * 100).first()[0]
check("silver", "service_line", "zero_pay_rate_plausible",
      1 <= zero_rate <= 40, f"{zero_rate:.1f}%", "1%-40%", severity="warn")

check("silver", "claim_header", "row_count", True, t, "n/a", severity="info")

flush()

failed = [r for r in _results if not r[5] and r[4] == "critical"]
if failed:
    raise Exception(f"{len(failed)} critical check(s) failed: {[r[3] for r in failed]}")
print("all critical checks passed")